In [68]:
import os
import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine, text, inspect
import psycopg2
from dotenv import load_dotenv

load_dotenv()  

PG_HOST = os.environ['POSTGRES_HOST']
PG_PORT = os.environ.get('POSTGRES_PORT', '5432')
PG_DB = os.environ['POSTGRES_DB']
PG_USER = os.environ['POSTGRES_USER']
PG_PASS = os.environ['POSTGRES_PASSWORD']


In [69]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine.url import URL

def get_engine(
    user: str, password: str, host: str, port: int, dbname: str,
    pool_size: int = 5, max_overflow: int = 10, pool_timeout: int = 30
):
    url = URL.create(
        drivername="postgresql+psycopg2",
        username=user,
        password=password,
        host=host,
        port=port,
        database=dbname
    )
    engine = create_engine(
        url,
        pool_size=pool_size,
        max_overflow=max_overflow,
        pool_timeout=pool_timeout,
        future=True # nutze moderne SQLAlchemy APIs
    )
    return engine

engine = get_engine(PG_USER, PG_PASS, PG_HOST, int(PG_PORT), PG_DB)

In [70]:
with engine.connect() as conn:
    res = conn.execute(text("SELECT 1"))
    print(res.scalar())  # sollte 1 ausgeben

1


In [71]:
def read_table_to_df(sql: str, engine):
    with engine.connect() as conn:
        return pd.read_sql(sql, conn)

df = read_table_to_df("SELECT * FROM bank_customers;", engine)
print(df.shape)
df.head()

(10000, 12)


,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [72]:
# Series für verschiedene Spalten erstellen
customer_ids = df['customer_id']
products_numbers = df['products_number']
ages = df['age']
balances = df['balance']
countries = df['country']
salaries = df['estimated_salary']
credit_scores = df['credit_score']
tenures = df['tenure']

# Mit customer_id Series arbeiten
print("=== CUSTOMER ID ANALYSIS ===")
print(f"Anzahl Kunden: {len(customer_ids)}")
print(f"Erste 10 IDs: {customer_ids.head(10).tolist()}")
print(f"Kleinste ID: {customer_ids.min()}")
print(f"Größte ID: {customer_ids.max()}")

# Mit age Series arbeiten
print("\n=== AGE ANALYSIS ===")
print(f"Durchschnittsalter: {ages.mean():.1f} Jahre")
print(f"Jüngster Kunde: {ages.min()} Jahre")
print(f"Ältester Kunde: {ages.max()} Jahre")
print(f"Altersverteilung:\n{ages.value_counts().sort_index()}")

# Mit balance Series arbeiten
print("\n=== BALANCE ANALYSIS ===")
print(f"Durchschnittsbalance: {balances.mean():.2f} €")
print(f"Höchster Kontostand: {balances.max():.2f} €")
rich_customers = balances[balances > 100000]
print(f"Kunden mit >100.000 €: {len(rich_customers)}")

# Mit country Series arbeiten
print("\n=== COUNTRY ANALYSIS ===")
print(f"Länderverteilung:\n{countries.value_counts()}")
print(f"Anzahl verschiedener Länder: {countries.nunique()}")

=== CUSTOMER ID ANALYSIS ===
Anzahl Kunden: 10000
Erste 10 IDs: [15634602, 15647311, 15619304, 15701354, 15737888, 15574012, 15592531, 15656148, 15792365, 15592389]
Kleinste ID: 15565701
Größte ID: 15815690

=== AGE ANALYSIS ===
Durchschnittsalter: 38.9 Jahre
Jüngster Kunde: 18 Jahre
Ältester Kunde: 92 Jahre
Altersverteilung:
age
18    22
19    27
20    40
21    53
22    84
      ..
83     1
84     2
85     1
88     1
92     2
Name: count, Length: 70, dtype: int64

=== BALANCE ANALYSIS ===
Durchschnittsbalance: 76485.89 €
Höchster Kontostand: 250898.09 €
Kunden mit >100.000 €: 4799

=== COUNTRY ANALYSIS ===
Länderverteilung:
country
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64
Anzahl verschiedener Länder: 3


In [73]:
def format_balances(balances):
    formated_balances = pd.cut(balances, bins=10, labels=[f"cluster_{i}" for i in range(1, 11)])
    return formated_balances

print(format_balances(balances).head)
print(f"Vermögensverteilung:\n{format_balances(balances).value_counts().sort_index()}")

<bound method NDFrame.head of 0       cluster_1
1       cluster_4
2       cluster_7
3       cluster_1
4       cluster_6
          ...    
9995    cluster_1
9996    cluster_3
9997    cluster_1
9998    cluster_3
9999    cluster_6
Name: balance, Length: 10000, dtype: category
Categories (10, object): ['cluster_1' < 'cluster_2' < 'cluster_3' < 'cluster_4' ... 'cluster_7' < 'cluster_8' < 'cluster_9' < 'cluster_10']>
Vermögensverteilung:
balance
cluster_1     3623
cluster_2       69
cluster_3      360
cluster_4     1173
cluster_5     2081
cluster_6     1747
cluster_7      729
cluster_8      186
cluster_9       30
cluster_10       2
Name: count, dtype: int64


In [74]:
def format_ages(ages):
    bins = [17, 30, 45, 55, 65, 100]  
    labels = [
        'Age_Young_Adults',           #(18-29)
        'Age_Adults_in_their_Prime',  #(30-44)
        'Age_Middle_aged',            #(45-54)
        'Age_Pre_retirees',           #(55-64)
        'Age_Young_Seniors'           #(65-77)
    ]
    
    formated_ages = pd.cut(ages, bins=bins, labels=labels, right=False)
    return formated_ages

print(f"Länge: {len(format_salaries(salaries))}")
print(format_ages(ages).head)
print(f"Altersverteilung:\n{format_ages(ages).value_counts().sort_index()}")

<bound method NDFrame.head of 0       Age_Adults_in_their_Prime
1       Age_Adults_in_their_Prime
2       Age_Adults_in_their_Prime
3       Age_Adults_in_their_Prime
4       Age_Adults_in_their_Prime
                  ...            
9995    Age_Adults_in_their_Prime
9996    Age_Adults_in_their_Prime
9997    Age_Adults_in_their_Prime
9998    Age_Adults_in_their_Prime
9999             Age_Young_Adults
Name: age, Length: 10000, dtype: category
Categories (5, object): ['Age_Young_Adults' < 'Age_Adults_in_their_Prime' < 'Age_Middle_aged' < 'Age_Pre_retirees' < 'Age_Young_Seniors']>
Altersverteilung:
age
Age_Young_Adults             1641
Age_Adults_in_their_Prime    6019
Age_Middle_aged              1458
Age_Pre_retirees              600
Age_Young_Seniors             282
Name: count, dtype: int64


In [80]:
def format_salaries(salaries):
    bins = [0.0, 1000.0, 10000.0, 30000.0, 60000.0, 100000.0, 150000.0, 200000.0]
    labels = [
        'Salarie_Very_low',       # (0-1k)
        'Salarie_Low',            # (1k-10k)
        'Salarie_Below_average',  # (10k-30k)
        'Salarie_Average',        # (30k-60k)
        'Salarie_Above_average',  # (60k-100k)
        'Salarie_High',           # (100k-150k)
        'Salarie_Very_high'       # (150k-200k)
    ]
    
    formatted_salaries = pd.cut(salaries, bins=bins, labels=labels, right=False)
    return formatted_salaries

print(format_salaries(salaries).head())  
print(salaries.head())

print(f"Gehaltsverteilung:\n{format_salaries(salaries).value_counts().sort_index()}")

print(f"Länge: {len(format_salaries(salaries))}")
print(f"Durchschnittseinkommen: {salaries.mean():.1f} Euro")
print(f"Ärmster Kunde: {salaries.min()} Euro")
print(f"Reichster Kunde: {salaries.max()} Euro")
print(f"Gehaltsverteilung:\n{salaries.value_counts().sort_index()}")

0             Salarie_High
1             Salarie_High
2             Salarie_High
3    Salarie_Above_average
4    Salarie_Above_average
Name: estimated_salary, dtype: category
Categories (7, object): ['Salarie_Very_low' < 'Salarie_Low' < 'Salarie_Below_average' < 'Salarie_Average' < 'Salarie_Above_average' < 'Salarie_High' < 'Salarie_Very_high']
0    101348.88
1    112542.58
2    113931.57
3     93826.63
4     79084.10
Name: estimated_salary, dtype: float64
Gehaltsverteilung:
estimated_salary
Salarie_Very_low           59
Salarie_Low               449
Salarie_Below_average     970
Salarie_Average          1483
Salarie_Above_average    2029
Salarie_High             2555
Salarie_Very_high        2455
Name: count, dtype: int64
Länge:
10000
Durchschnittseinkommen: 100090.2 Euro
Ärmster Kunde: 11.58 Euro
Reichster Kunde: 199992.48 Euro
Gehaltsverteilung:
estimated_salary
11.58        1
90.07        1
91.75        1
96.27        1
106.67       1
            ..
199909.32    1
199929.17    1
19

In [79]:
def format_credit_scores(credit_scores):
    bins = [0, 580, 670, 740, 800, 851]
    labels = [
        'Credit_Score_Poor',               # 300-579
        'Credit_Score_Fair',               # 580-669
        'Credit_Score_Good',               # 670-739
        'Credit_Score_Very_Good',          # 740-799
        'Credit_Score_Excellent'           # 800-850
    ]
    
    formatted_credit_scores = pd.cut(credit_scores, bins=bins, labels=labels, right=False)
    return formatted_credit_scores

print(format_credit_scores(credit_scores).head())  
print(credit_scores.head())

print(f"Verteilung:\n{format_credit_scores(credit_scores).value_counts().sort_index()}")
print(f"Länge:\n{len(format_credit_scores(credit_scores))}")
print(f"Verteilung:\n{credit_scores.value_counts().sort_index()}")

0         Credit_Score_Fair
1         Credit_Score_Fair
2         Credit_Score_Poor
3         Credit_Score_Good
4    Credit_Score_Excellent
Name: credit_score, dtype: category
Categories (5, object): ['Credit_Score_Poor' < 'Credit_Score_Fair' < 'Credit_Score_Good' < 'Credit_Score_Very_Good' < 'Credit_Score_Excellent']
0    619
1    608
2    502
3    699
4    850
Name: credit_score, dtype: int64
Verteilung:
credit_score
Credit_Score_Poor         2362
Credit_Score_Fair         3331
Credit_Score_Good         2428
Credit_Score_Very_Good    1224
Credit_Score_Excellent     655
Name: count, dtype: int64
Länge:
10000
Durchschnitt: 650.5 
Min: 350 
Max: 850 
Verteilung:
credit_score
350      5
351      1
358      1
359      1
363      1
      ... 
846      5
847      6
848      5
849      8
850    233
Name: count, Length: 460, dtype: int64


In [84]:
def format_tenures(tenures):
    formated_tenures = "Tenure_" + tenures.astype(str)
    return formated_tenures

print(format_credit_scores(credit_scores).head())  
print(credit_scores.head())

print(f"Verteilung:\n{format_tenures(tenures).value_counts().sort_index()}")
print(f"Länge:\n{len(format_tenures(tenures))}")
print(f"Verteilung:\n{format_tenures(tenures).value_counts().sort_index()}")

0         Credit_Score_Fair
1         Credit_Score_Fair
2         Credit_Score_Poor
3         Credit_Score_Good
4    Credit_Score_Excellent
Name: credit_score, dtype: category
Categories (5, object): ['Credit_Score_Poor' < 'Credit_Score_Fair' < 'Credit_Score_Good' < 'Credit_Score_Very_Good' < 'Credit_Score_Excellent']
0    619
1    608
2    502
3    699
4    850
Name: credit_score, dtype: int64
Verteilung:
tenure
Tenure_0      413
Tenure_1     1035
Tenure_10     490
Tenure_2     1048
Tenure_3     1009
Tenure_4      989
Tenure_5     1012
Tenure_6      967
Tenure_7     1028
Tenure_8     1025
Tenure_9      984
Name: count, dtype: int64
Länge:
10000
Verteilung:
tenure
Tenure_0      413
Tenure_1     1035
Tenure_10     490
Tenure_2     1048
Tenure_3     1009
Tenure_4      989
Tenure_5     1012
Tenure_6      967
Tenure_7     1028
Tenure_8     1025
Tenure_9      984
Name: count, dtype: int64


In [85]:
print(type(df))

<class 'pandas.core.frame.DataFrame'>
